# 02 - RNN From Scratch: Intuition

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the core idea of a Recurrent Neural Network: a **hidden state** that carries information across time steps
- Write the RNN cell equation from memory and implement it from scratch in PyTorch (no `nn.RNN`)
- Describe how an RNN is **unrolled through time** and connect this to backpropagation through time (BPTT)
- Demonstrate the **vanishing gradient problem** by measuring gradient magnitudes over long sequences
- Explain the concept of **bidirectional RNNs** and when they are useful

## Prerequisites

- Notebook 01: Sequence Data and Embeddings
- PyTorch basics: tensors, `nn.Module`, autograd
- Linear algebra: matrix multiplication, `tanh` activation

## Table of Contents

1. [The Core Idea: Hidden State](#1-the-core-idea-hidden-state)
2. [The RNN Cell Equation](#2-the-rnn-cell-equation)
3. [Code: RNN Cell From Scratch](#3-code-rnn-cell-from-scratch)
4. [Code: Processing a Sequence Step by Step](#4-code-processing-a-sequence-step-by-step)
5. [Unrolling Through Time](#5-unrolling-through-time)
6. [The Vanishing Gradient Problem](#6-the-vanishing-gradient-problem)
7. [Code: Demonstrate Vanishing Gradients](#7-code-demonstrate-vanishing-gradients)
8. [Bidirectional RNN (Concept)](#8-bidirectional-rnn-concept)
9. [Exercise: Character-Level RNN for Tiny Sequence Prediction](#9-exercise-character-level-rnn-for-tiny-sequence-prediction)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. The Core Idea: Hidden State

A **feedforward** network processes each input independently. It has no memory of previous inputs.

A **Recurrent Neural Network (RNN)** introduces a **hidden state** $h_t$ that is updated at every time step and passed forward to the next step. This hidden state acts as the network's **memory**.

### Intuition

Think of reading a sentence word by word:

- After reading "The", you have some expectation about what follows
- After reading "The cat", your understanding is updated
- After reading "The cat sat on the", you can predict "mat" or "floor"

The hidden state $h_t$ captures this evolving understanding:

$$h_0 \xrightarrow{x_1} h_1 \xrightarrow{x_2} h_2 \xrightarrow{x_3} h_3 \cdots$$

At each time step $t$, the RNN:
1. Takes the current input $x_t$
2. Combines it with the previous hidden state $h_{t-1}$
3. Produces a new hidden state $h_t$

In [ ]:
# Visualize the RNN concept: hidden state evolving over time
fig, ax = plt.subplots(figsize=(12, 4))

words = ["The", "cat", "sat", "on", "the", "mat"]
n = len(words)

for i in range(n):
    # Input box
    ax.add_patch(plt.Rectangle((i * 2, 0), 1.2, 0.6, fill=True,
                                facecolor='#AED6F1', edgecolor='black', linewidth=1.5))
    ax.text(i * 2 + 0.6, 0.3, f'$x_{i+1}$={words[i]}', ha='center', va='center', fontsize=9)

    # Hidden state box
    ax.add_patch(plt.Rectangle((i * 2, 1.2), 1.2, 0.6, fill=True,
                                facecolor='#F9E79F', edgecolor='black', linewidth=1.5))
    ax.text(i * 2 + 0.6, 1.5, f'$h_{i+1}$', ha='center', va='center', fontsize=11)

    # Arrow: input -> hidden
    ax.annotate('', xy=(i * 2 + 0.6, 1.2), xytext=(i * 2 + 0.6, 0.6),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.5))

    # Arrow: hidden -> next hidden
    if i < n - 1:
        ax.annotate('', xy=((i + 1) * 2, 1.5), xytext=(i * 2 + 1.2, 1.5),
                    arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

# Initial hidden state
ax.text(-0.5, 1.5, '$h_0$', ha='center', va='center', fontsize=11, color='#E74C3C')
ax.annotate('', xy=(0, 1.5), xytext=(-0.2, 1.5),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

ax.set_xlim(-1, n * 2 + 0.5)
ax.set_ylim(-0.3, 2.3)
ax.set_title('RNN: Hidden State Evolves Over Time Steps', fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 2. The RNN Cell Equation

The standard (Elman) RNN cell computes:

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$$

where:

| Symbol | Shape | Description |
|--------|-------|-------------|
| $x_t$ | $(d_{\text{in}},)$ | Input at time step $t$ |
| $h_{t-1}$ | $(d_{\text{hidden}},)$ | Previous hidden state |
| $W_{xh}$ | $(d_{\text{hidden}}, d_{\text{in}})$ | Input-to-hidden weight matrix |
| $W_{hh}$ | $(d_{\text{hidden}}, d_{\text{hidden}})$ | Hidden-to-hidden weight matrix |
| $b$ | $(d_{\text{hidden}},)$ | Bias vector |
| $h_t$ | $(d_{\text{hidden}},)$ | New hidden state |

### Why `tanh`?

- Outputs range $[-1, 1]$, keeping hidden states bounded
- Zero-centered (unlike sigmoid), which helps gradient flow
- Standard choice for vanilla RNNs (LSTMs/GRUs use a mix of `tanh` and `sigmoid`)

---
## 3. Code: RNN Cell From Scratch

Let us implement the RNN cell equation manually using only basic PyTorch operations -- no `nn.RNN` or `nn.RNNCell`.

In [ ]:
class RNNCellFromScratch(nn.Module):
    """Vanilla RNN cell implemented from scratch.
    
    h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b)
    """
    
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Learnable parameters
        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.b = nn.Parameter(torch.zeros(hidden_size))
    
    def forward(self, x_t, h_prev):
        """One step of the RNN.
        
        Args:
            x_t: Input at current time step, shape (batch, input_size)
            h_prev: Previous hidden state, shape (batch, hidden_size)
        
        Returns:
            h_t: New hidden state, shape (batch, hidden_size)
        """
        # h_t = tanh(W_xh @ x_t + W_hh @ h_prev + b)
        h_t = torch.tanh(
            x_t @ self.W_xh.T + h_prev @ self.W_hh.T + self.b
        )
        return h_t


# Test our implementation
set_global_seed(42)

input_size = 4
hidden_size = 8
batch_size = 2

cell = RNNCellFromScratch(input_size, hidden_size)

# Create dummy input and initial hidden state
x_t = torch.randn(batch_size, input_size)
h_prev = torch.zeros(batch_size, hidden_size)

h_new = cell(x_t, h_prev)

print(f"Input shape:          {x_t.shape}")
print(f"Previous hidden:      {h_prev.shape}")
print(f"New hidden state:     {h_new.shape}")
print(f"Hidden values (sample): {h_new[0, :4].data}")
print(f"Values in [-1, 1]:   {(h_new.abs() <= 1.0).all().item()}  (tanh output)")

In [ ]:
# Verify against PyTorch's built-in nn.RNNCell
set_global_seed(42)

# Create both cells with same weights
our_cell = RNNCellFromScratch(input_size=4, hidden_size=8)
pytorch_cell = nn.RNNCell(input_size=4, hidden_size=8)

# Copy our weights into PyTorch cell
with torch.no_grad():
    pytorch_cell.weight_ih.copy_(our_cell.W_xh)
    pytorch_cell.weight_hh.copy_(our_cell.W_hh)
    pytorch_cell.bias_ih.zero_()
    pytorch_cell.bias_hh.copy_(our_cell.b)

x = torch.randn(2, 4)
h0 = torch.zeros(2, 8)

h_ours = our_cell(x, h0)
h_pytorch = pytorch_cell(x, h0)

print(f"Our output:     {h_ours[0, :4].data}")
print(f"PyTorch output: {h_pytorch[0, :4].data}")
print(f"Close match:    {torch.allclose(h_ours, h_pytorch, atol=1e-6)}")

---
## 4. Code: Processing a Sequence Step by Step

Now let us process an entire sequence by running the RNN cell at each time step and collecting hidden states.

In [ ]:
class RNNFromScratch(nn.Module):
    """Full RNN layer that processes an entire sequence."""
    
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = RNNCellFromScratch(input_size, hidden_size)
    
    def forward(self, x, h0=None):
        """Process entire sequence.
        
        Args:
            x: Input sequence, shape (batch, seq_len, input_size)
            h0: Initial hidden state, shape (batch, hidden_size)
        
        Returns:
            outputs: Hidden states at all time steps, shape (batch, seq_len, hidden_size)
            h_n: Final hidden state, shape (batch, hidden_size)
        """
        batch_size, seq_len, _ = x.shape
        
        if h0 is None:
            h0 = torch.zeros(batch_size, self.hidden_size, device=x.device)
        
        h_t = h0
        outputs = []
        
        for t in range(seq_len):
            h_t = self.cell(x[:, t, :], h_t)  # (batch, hidden_size)
            outputs.append(h_t)
        
        # Stack: list of (batch, hidden) -> (batch, seq_len, hidden)
        outputs = torch.stack(outputs, dim=1)
        return outputs, h_t


set_global_seed(42)

# Create a synthetic sequence
batch_size = 1
seq_len = 6
input_size = 4
hidden_size = 8

rnn = RNNFromScratch(input_size, hidden_size)
x_seq = torch.randn(batch_size, seq_len, input_size)

outputs, h_final = rnn(x_seq)

print(f"Input shape:       {x_seq.shape}  (batch, seq_len, input_size)")
print(f"All hidden states: {outputs.shape}  (batch, seq_len, hidden_size)")
print(f"Final hidden:      {h_final.shape}  (batch, hidden_size)")
print(f"\nFinal hidden matches last output: {torch.allclose(outputs[:, -1, :], h_final)}")

In [ ]:
# Visualize hidden state evolution across time steps
set_global_seed(42)

rnn_vis = RNNFromScratch(input_size=4, hidden_size=16)
x_vis = torch.randn(1, 20, 4)  # 20 time steps
outputs_vis, _ = rnn_vis(x_vis)

# Extract hidden states: (seq_len, hidden_size)
hidden_states = outputs_vis.squeeze(0).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of hidden state evolution
im = axes[0].imshow(hidden_states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Hidden Dimension')
axes[0].set_title('Hidden State Evolution (Heatmap)', fontweight='bold')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Line plot of selected hidden dimensions
for dim in [0, 3, 7, 12]:
    axes[1].plot(hidden_states[:, dim], label=f'dim {dim}', linewidth=1.5)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Activation Value')
axes[1].set_title('Selected Hidden Dimensions Over Time', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.show()

print("Each column is the hidden state at one time step.")
print("Notice how the hidden state evolves as each input is processed.")

---
## 5. Unrolling Through Time

When we "unroll" an RNN, we visualize the same cell replicated at each time step, with shared weights:

$$h_1 = \tanh(W_{xh}x_1 + W_{hh}h_0 + b)$$
$$h_2 = \tanh(W_{xh}x_2 + W_{hh}h_1 + b)$$
$$h_3 = \tanh(W_{xh}x_3 + W_{hh}h_2 + b)$$
$$\vdots$$

### Key points about unrolling

- **Weight sharing**: The same $W_{xh}$, $W_{hh}$, and $b$ are used at every time step
- **Backpropagation Through Time (BPTT)**: Gradients flow backward through the unrolled graph
- **Computational graph grows** with sequence length, which is why long sequences are expensive

### BPTT gradient flow

To compute $\frac{\partial L}{\partial W_{hh}}$, the gradient must flow back through every time step:

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}$$

Each $\frac{\partial h_t}{\partial h_{t-1}}$ involves multiplying by $W_{hh}$ and the derivative of $\tanh$. Over many steps, this leads to the **vanishing gradient problem**.

---
## 6. The Vanishing Gradient Problem

The gradient of the loss with respect to an early hidden state $h_k$ involves a product of Jacobians:

$$\frac{\partial h_t}{\partial h_k} = \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}} = \prod_{i=k+1}^{t} \text{diag}(1 - h_i^2) \cdot W_{hh}$$

### Why gradients vanish

- $\tanh'(x) = 1 - \tanh^2(x) \in (0, 1]$, so the derivative is always $\leq 1$
- If the largest singular value of $W_{hh}$ is $< 1$, the product **shrinks exponentially**
- After $T$ time steps, the gradient magnitude $\approx \sigma_{\max}^T \to 0$

### Consequence

- The RNN cannot learn **long-range dependencies** (e.g., subject-verb agreement across 20+ words)
- Early time steps receive negligible gradient updates
- This is the primary motivation for LSTM and GRU architectures (next notebook)

---
## 7. Code: Demonstrate Vanishing Gradients

Let us measure how gradient magnitudes decay as they flow back through time.

In [ ]:
set_global_seed(42)

# Create a simple RNN and process a long sequence
seq_len = 50
input_size = 10
hidden_size = 32

rnn_grad = RNNFromScratch(input_size, hidden_size)
x_long = torch.randn(1, seq_len, input_size)

# Process the sequence -- retain grad on hidden states
h_t = torch.zeros(1, hidden_size, requires_grad=True)
hidden_states_list = [h_t]

for t in range(seq_len):
    h_t = rnn_grad.cell(x_long[:, t, :], h_t)
    hidden_states_list.append(h_t)

# Compute a simple loss from the final hidden state
loss = hidden_states_list[-1].sum()

# Backpropagate
loss.backward()

# Measure gradient magnitude at each time step
grad_norms = []
for t in range(seq_len + 1):
    if hidden_states_list[t].grad is not None:
        grad_norms.append(hidden_states_list[t].grad.norm().item())
    else:
        # Compute gradient manually via autograd.grad for intermediate nodes
        grad = torch.autograd.grad(
            loss, hidden_states_list[t], retain_graph=True
        )[0]
        grad_norms.append(grad.norm().item())

print(f"Gradient norm at t=0 (earliest): {grad_norms[0]:.6f}")
print(f"Gradient norm at t={seq_len//2} (middle):   {grad_norms[seq_len//2]:.6f}")
print(f"Gradient norm at t={seq_len} (latest):  {grad_norms[seq_len]:.6f}")
print(f"\nRatio (earliest / latest): {grad_norms[0] / (grad_norms[seq_len] + 1e-10):.6f}")

In [ ]:
# Plot gradient norms over time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(range(seq_len + 1), grad_norms, 'b-o', markersize=3, linewidth=1.5)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Gradient Norm')
axes[0].set_title('Gradient Norms Over Time (Linear Scale)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Log scale
axes[1].semilogy(range(seq_len + 1), grad_norms, 'r-o', markersize=3, linewidth=1.5)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Gradient Norm (log scale)')
axes[1].set_title('Gradient Norms Over Time (Log Scale)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation: Gradients at early time steps are orders of magnitude")
print("smaller than at later steps. The RNN effectively 'forgets' early inputs.")

In [ ]:
# Demonstrate: repeated multiplication by W_hh causes exponential decay/growth
set_global_seed(42)

hidden_size_demo = 16

# Case 1: W_hh with singular values < 1 (vanishing)
W_vanish = torch.randn(hidden_size_demo, hidden_size_demo) * 0.5

# Case 2: W_hh with singular values > 1 (exploding)
W_explode = torch.randn(hidden_size_demo, hidden_size_demo) * 1.5

steps = 50
norms_vanish = []
norms_explode = []

v = torch.randn(hidden_size_demo)
v_vanish = v.clone()
v_explode = v.clone()

for t in range(steps):
    v_vanish = W_vanish @ v_vanish
    v_explode = W_explode @ v_explode
    norms_vanish.append(v_vanish.norm().item())
    norms_explode.append(min(v_explode.norm().item(), 1e15))  # cap for plotting

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(norms_vanish, 'b-o', markersize=3)
axes[0].set_title('Vanishing: $\\|W^t v\\|$ with small $\\sigma(W)$', fontweight='bold')
axes[0].set_xlabel('Multiplications')
axes[0].set_ylabel('Vector Norm (log scale)')
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(norms_explode, 'r-o', markersize=3)
axes[1].set_title('Exploding: $\\|W^t v\\|$ with large $\\sigma(W)$', fontweight='bold')
axes[1].set_xlabel('Multiplications')
axes[1].set_ylabel('Vector Norm (log scale)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"After {steps} multiplications:")
print(f"  Vanishing case: norm = {norms_vanish[-1]:.2e}")
print(f"  Exploding case: norm = {norms_explode[-1]:.2e}")

---
## 8. Bidirectional RNN (Concept)

A standard RNN only processes the sequence **left to right**. A **bidirectional RNN** processes it in both directions:

- **Forward RNN**: $h_t^{\rightarrow} = \tanh(W_{xh}^{\rightarrow}x_t + W_{hh}^{\rightarrow}h_{t-1}^{\rightarrow} + b^{\rightarrow})$
- **Backward RNN**: $h_t^{\leftarrow} = \tanh(W_{xh}^{\leftarrow}x_t + W_{hh}^{\leftarrow}h_{t+1}^{\leftarrow} + b^{\leftarrow})$

The outputs are concatenated: $h_t = [h_t^{\rightarrow}; h_t^{\leftarrow}]$, giving a vector of size $2 \times d_{\text{hidden}}$.

### When to use bidirectional RNNs

| Use Case | Bidirectional? | Why? |
|----------|:-----------:|------|
| Text classification | Yes | Full sentence available upfront |
| Named entity recognition | Yes | Context from both sides helps |
| Language modeling (next word) | No | Cannot look at future words |
| Real-time speech recognition | No | Future audio not yet available |
| Machine translation (encoder) | Yes | Encoder sees full input |

In [ ]:
# Quick demo of bidirectional RNN using PyTorch's built-in
set_global_seed(42)

rnn_uni = nn.RNN(input_size=4, hidden_size=8, batch_first=True, bidirectional=False)
rnn_bi = nn.RNN(input_size=4, hidden_size=8, batch_first=True, bidirectional=True)

x_demo = torch.randn(1, 6, 4)  # batch=1, seq_len=6, input=4

out_uni, _ = rnn_uni(x_demo)
out_bi, _ = rnn_bi(x_demo)

print(f"Unidirectional output shape: {out_uni.shape}  (hidden_size = 8)")
print(f"Bidirectional output shape:  {out_bi.shape}  (hidden_size = 2 x 8 = 16)")
print(f"\nBidirectional output at each step is the concatenation of")
print(f"forward and backward hidden states.")

---
## 9. Exercise: Character-Level RNN for Tiny Sequence Prediction

**Task:** Build a character-level RNN that learns to predict the next character in simple repeating patterns.

1. Use the pattern `"abcabcabc"` as training data
2. Create a character vocabulary and one-hot encode inputs
3. Use `RNNFromScratch` (or your own) to process the sequence
4. Add a linear output layer to predict the next character
5. Train for 200 epochs and report the loss
6. Test: given `"ab"`, does the model predict `"c"`?

```python
# Your code here
set_global_seed(42)

# Step 1: Define the pattern and character vocabulary
# pattern = "abcabcabc"
# chars = sorted(set(pattern))
# char_to_idx = ...

# Step 2: Create input (chars[:-1]) and target (chars[1:]) sequences
# One-hot encode inputs

# Step 3: Build model: RNNFromScratch + nn.Linear

# Step 4: Train

# Step 5: Test prediction
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

# Step 1: Pattern and vocabulary
pattern = "abcabcabc"
chars = sorted(set(pattern))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Pattern: '{pattern}'")
print(f"Vocabulary: {char_to_idx}")
print(f"Vocab size: {vocab_size}")

# Step 2: Create input and target sequences
input_chars = pattern[:-1]   # 'abcabcab'
target_chars = pattern[1:]   # 'bcabcabc'

# One-hot encode inputs
input_indices = [char_to_idx[c] for c in input_chars]
target_indices = [char_to_idx[c] for c in target_chars]

x_onehot = torch.zeros(1, len(input_indices), vocab_size)
for t, idx in enumerate(input_indices):
    x_onehot[0, t, idx] = 1.0

y_target = torch.tensor(target_indices).unsqueeze(0)  # (1, seq_len)

print(f"\nInput:  {input_chars} -> indices {input_indices}")
print(f"Target: {target_chars} -> indices {target_indices}")
print(f"Input tensor shape:  {x_onehot.shape}")
print(f"Target tensor shape: {y_target.shape}")

In [ ]:
# Step 3: Build model
set_global_seed(42)

class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = RNNFromScratch(vocab_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        outputs, _ = self.rnn(x)      # (batch, seq_len, hidden)
        logits = self.fc(outputs)      # (batch, seq_len, vocab_size)
        return logits

model = CharRNN(vocab_size=vocab_size, hidden_size=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Step 4: Train
losses = []
for epoch in range(200):
    model.train()
    logits = model(x_onehot)  # (1, seq_len, vocab_size)
    
    # Reshape for CrossEntropyLoss: (batch*seq_len, vocab_size) vs (batch*seq_len,)
    loss = loss_fn(logits.view(-1, vocab_size), y_target.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Character-Level RNN Training Loss', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Step 5: Test prediction
model.eval()

# Given "ab", predict next character
test_input = "ab"
test_indices = [char_to_idx[c] for c in test_input]
x_test = torch.zeros(1, len(test_indices), vocab_size)
for t, idx in enumerate(test_indices):
    x_test[0, t, idx] = 1.0

with torch.no_grad():
    logits = model(x_test)
    # Take the prediction at the last time step
    probs = torch.softmax(logits[0, -1, :], dim=0)
    predicted_idx = probs.argmax().item()
    predicted_char = idx_to_char[predicted_idx]

print(f"Input: '{test_input}'")
print(f"Predicted next char: '{predicted_char}'")
print(f"Probabilities: {dict(zip(chars, probs.tolist()))}")
print(f"\nCorrect! The pattern 'abc' means after 'ab' comes 'c'." 
      if predicted_char == 'c' else "\nIncorrect -- try more training epochs.")

---
## 10. Common Mistakes & Debugging Tips

**1. Forgetting to initialize hidden state to zeros**
- If $h_0$ is not provided, initialize it as a zero tensor with the correct shape `(batch, hidden_size)`.
- Using `None` or a random tensor can cause inconsistent results.

**2. Shape mismatch between input and weights**
- Input `x_t` should be `(batch, input_size)`, not `(input_size,)` -- always include the batch dimension.
- Use `x.unsqueeze(0)` if processing a single sample.

**3. Confusing sequence dimension ordering**
- PyTorch's `nn.RNN` default is `(seq_len, batch, features)`. Use `batch_first=True` for `(batch, seq_len, features)`.
- Our from-scratch implementation uses `batch_first=True` convention.

**4. Not detaching hidden state between sequences during training**
- When training on multiple chunks of a long sequence, detach the hidden state between chunks: `h = h.detach()`.
- Otherwise the computational graph grows unboundedly and you run out of memory.

**5. Gradient explosion**
- If loss becomes `NaN`, apply **gradient clipping**: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)`.
- This is standard practice for RNN training.

**6. Expecting vanilla RNN to learn long-range dependencies**
- As we showed, vanilla RNNs suffer from vanishing gradients.
- For sequences longer than ~20 steps, use LSTM or GRU (next notebook).

---

**Next notebook:** [03 - LSTM and GRU: Why They Work](03_LSTM_and_GRU_Why_They_Work.ipynb) -- we learn how gating mechanisms solve the vanishing gradient problem.